# HEAFormer: Site-Occupancy Transformer for Short-Range Order Prediction

**Authors:** Ishraque Zaman Borshon, Prof. Vitaliy Yurkiv  
**Institution:** University of Arizona, Aerospace & Mechanical Engineering  

---

This notebook demonstrates the full HEAFormer pipeline on real (MC-generated) data:

1. FCC supercell generation and Warren-Cowley SRO computation  
2. Site-occupancy tokenisation  
3. Sklearn baseline models  
4. MLM pre-training of the transformer encoder  
5. Physics-informed SRO fine-tuning  
6. Evaluation and visualisation  
7. Attention weight interpretability  
8. Physics consistency check  

**Runtime:** ~3 minutes on a modern CPU (quick mode).  
Set `QUICK = False` for the full experiment (~20 min).


In [1]:
import sys, os
# Make sure we can import from the repo root
REPO = os.path.abspath(os.path.join(os.getcwd(), '..'))
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.chdir(REPO)
print('Working dir:', os.getcwd())

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

QUICK = True   # <-- set False for full run
SEED  = 42
np.random.seed(SEED)
print(f'Quick mode: {QUICK}')

Working dir: d:\Docker_Sandbox\HEA_former\hea_transformer_full\hea_transformer
Quick mode: True


## 1. Dataset Generation

We generate FCC supercells of the Cantor alloy (CrMnFeCoNi) across five
ordering scenarios using Metropolis Monte Carlo with pairwise interaction
matrices.  Warren-Cowley SRO parameters are computed as regression targets.

**Why 3×3×3?** The 2-shell SRO computation requires the box side to be
≥ 3× the 2NN distance (3.6 Å for FCC-Ni), giving 10.8 Å.  A 2×2×2 box
(7.2 Å) causes minimum-image collapse that halves the 2NN count to 3.

In [2]:
from src.data.supercell_generator import build_dataset, N_SP, CANTOR, VOCAB
from src.physics.sro import nearest_neighbors

cfg = dict(
    nx=3, ny=3, nz=3,
    n_per_scenario = 30 if QUICK else 100,
    n_mc_steps     = 2000 if QUICK else 12000,
    scenarios      = ['random', 'ordering', 'cluster']
                     if QUICK else
                     ['random', 'ordering', 'cluster', 'mixed', 'CrMn_ordering'],
    seed=SEED,
)

ds = build_dataset(**cfg, n_shells=2, verbose=True)
print(f"\nDataset keys: {list(ds.keys())}")

[Dataset] 3x3x3 FCC supercell: 108 atoms
[Dataset] Building neighbor lists ... done  (1NN=12, 2NN=6)
[Dataset] 'random': 30 configs ... done  phases={1: 30}
[Dataset] 'ordering': 30 configs ... done  phases={2: 29, 1: 1}
[Dataset] 'cluster': 30 configs ... done  phases={1: 30}

[Dataset] Total=90 | atoms=108 | SRO_dim=50
          alpha range=[-1.612, 0.714]
          phase counts={np.int64(1): np.int64(61), np.int64(2): np.int64(29)}

Dataset keys: ['occupancy', 'coords_frac', 'box', 'shells', 'dists', 'labels', 'alpha_flat', 'phase_labels', 'mean_alpha', 'scenario_ids', 'meta']


### 1a. Verify neighbour lists (critical correctness check)

In [3]:
nn1 = [len(ds['shells'][i][0]) for i in range(ds['meta']['N'])]
nn2 = [len(ds['shells'][i][1]) for i in range(ds['meta']['N'])]
print(f"1NN:  min={min(nn1)}  max={max(nn1)}  mean={np.mean(nn1):.1f}  (ideal FCC = 12)")
print(f"2NN:  min={min(nn2)}  max={max(nn2)}  mean={np.mean(nn2):.1f}  (ideal FCC = 6)")
assert min(nn1) == 12 == max(nn1), 'ERROR: wrong 1NN count'
assert min(nn2) ==  6 == max(nn2), 'ERROR: wrong 2NN count'
print("[PASS] Neighbour lists are correct.")

1NN:  min=12  max=12  mean=12.0  (ideal FCC = 12)
2NN:  min=6  max=6  mean=6.0  (ideal FCC = 6)
[PASS] Neighbour lists are correct.


### 1b. Visualise ground-truth SRO matrices

In [4]:
from src.evaluation.plots import plot_sro_matrix

fig, axes = plt.subplots(1, min(3, len(cfg['scenarios'])),
                          figsize=(5.5*min(3,len(cfg['scenarios'])), 4.5))
if len(cfg['scenarios']) == 1:
    axes = [axes]

EL_COL = {'Cr':'#E64B35','Mn':'#4DBBD5','Fe':'#00A087','Co':'#3C5488','Ni':'#F39B7F'}

for ax, sc in zip(axes, cfg['scenarios'][:3]):
    si = cfg['scenarios'].index(sc)
    mask = ds['scenario_ids'] == si
    idx0 = np.where(mask)[0][0]
    A = ds['labels'][idx0]['alpha'][0]   # shell-1 SRO matrix
    ns = A.shape[0]
    vmax = max(abs(A).max(), 0.2)
    im = ax.imshow(A, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='equal')
    plt.colorbar(im, ax=ax, label=r'$\alpha_{ij}$')
    ax.set_xticks(range(ns)); ax.set_yticks(range(ns))
    ax.set_xticklabels(CANTOR, fontsize=9)
    ax.set_yticklabels(CANTOR, fontsize=9)
    for i in range(ns):
        for j in range(ns):
            ax.text(j, i, f'{A[i,j]:+.2f}', ha='center', va='center',
                    fontsize=7, color='white' if abs(A[i,j]) > 0.5*vmax else 'black')
    ax.set_title(f'Scenario: {sc}', fontsize=11)

fig.suptitle('Warren-Cowley SRO matrices (shell 1)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/nb_sro_matrices.png', dpi=130, bbox_inches='tight')
plt.show()
print("Figure saved to figures/nb_sro_matrices.png")

Figure saved to figures/nb_sro_matrices.png


## 2. Tokenisation

Each site becomes a factorised token:
- `token_id` : element index (or `[MASK]`)
- `features` : shell-resolved composition + coordination number + global composition + fractional position

In [5]:
from src.data.tokenizer import SiteOccupancyTokenizer, SequenceDataset

tok = SiteOccupancyTokenizer(n_shells=2)
print(f"Feature dimension : {tok.feat_dim}")
print(f"Feature names     : {tok.feature_names()}")

seq = SequenceDataset(ds, tok, mask_prob=0.15, seed=SEED)
print(f"\nDataset size      : {len(seq)} supercells")

s0 = seq[0]
print(f"Sample keys       : {list(s0.keys())}")
print(f"token_ids (first 8): {s0['token_ids'][:8]}")
print(f"Masked positions  : {np.where(s0['mask'])[0][:8]}")
print(f"features shape    : {s0['features'].shape}")
print(f"alpha_flat shape  : {s0['alpha_flat'].shape}")
print(f"phase_label       : {s0['phase_label']}")

Feature dimension : 20
Feature names     : ['sh1_Cr', 'sh1_Mn', 'sh1_Fe', 'sh1_Co', 'sh1_Ni', 'sh2_Cr', 'sh2_Mn', 'sh2_Fe', 'sh2_Co', 'sh2_Ni', 'cn1', 'cn2', 'glob_Cr', 'glob_Mn', 'glob_Fe', 'glob_Co', 'glob_Ni', 'fx', 'fy', 'fz']

Dataset size      : 90 supercells
Sample keys       : ['token_ids', 'true_ids', 'mask', 'features', 'x_global', 'alpha_flat', 'phase_label', 'mean_alpha', 'scenario_id']
token_ids (first 8): [2 3 0 2 2 2 4 5]
Masked positions  : [ 4  7 15 23 31 43 44 50]
features shape    : (108, 20)
alpha_flat shape  : (50,)
phase_label       : 1


In [6]:
# Visualise masking
from src.evaluation.plots import plot_tokenization

fig, axes = plt.subplots(2, 1, figsize=(14, 3.5))
n_show = min(28, ds['meta']['N'])

for row, (ids, title, use_mask) in enumerate([
    (ds['occupancy'][0], 'True site occupancy', False),
    (s0['token_ids'],    f'Masked input ({s0["mask"].sum()} masked)', True),
]):
    ax = axes[row]
    for i in range(n_show):
        t = int(ids[i])
        m = s0['mask'][i] if use_mask else False
        if m:
            color, label, ec, lw = '#B0BEC5', '[M]', 'red', 2.0
        elif 0 <= t < len(CANTOR):
            color, label, ec, lw = list(EL_COL.values())[t], CANTOR[t], 'k', 0.5
        else:
            color, label, ec, lw = '#CFD8DC', '?', 'k', 0.5
        ax.bar(i, 1, color=color, edgecolor=ec, linewidth=lw, width=0.9)
        ax.text(i, 0.5, label, ha='center', va='center',
                fontsize=6, fontweight='bold',
                color='white' if not m else 'red')
    ax.set_xlim(-0.5, n_show-0.5); ax.set_ylim(0,1); ax.set_yticks([])
    ax.set_xticks(range(n_show))
    ax.set_xticklabels([str(i) for i in range(n_show)], fontsize=5)
    ax.set_title(title, fontsize=10, loc='left')

handles = [mpatches.Patch(color=c, label=e) for e, c in EL_COL.items()]
axes[0].legend(handles=handles, loc='upper right', ncol=5, fontsize=7)
plt.tight_layout(pad=0.6)
plt.savefig('figures/nb_tokenization.png', dpi=130, bbox_inches='tight')
plt.show()

## 3. Train/Val/Test Split

In [7]:
rng = np.random.default_rng(SEED)
n   = len(seq)
perm = rng.permutation(n)
n_tr, n_va = int(0.70*n), int(0.15*n)
tr_idx = perm[:n_tr].tolist()
va_idx = perm[n_tr:n_tr+n_va].tolist()
te_idx = perm[n_tr+n_va:].tolist()
print(f"Train={len(tr_idx)}  Val={len(va_idx)}  Test={len(te_idx)}")

# Phase distribution in each split
from collections import Counter
for name, idx in [('Train', tr_idx), ('Val', va_idx), ('Test', te_idx)]:
    phases = [seq[i]['phase_label'] for i in idx]
    print(f"  {name}: {dict(Counter(phases))}")

Train=62  Val=13  Test=15
  Train: {1: 44, 2: 18}
  Val: {2: 6, 1: 7}
  Test: {2: 5, 1: 10}


## 4. Sklearn Baseline Models

In [8]:
from src.evaluation.baselines import run_all_baselines

print("Running baseline models...")
bl = run_all_baselines(seq, tr_idx, te_idx, verbose=True)

Running baseline models...
  [SRO] Ridge (comp) ... MAE=0.2083  R²=-0.1072
  [SRO] MLP (comp) ... MAE=0.2165  R²=-0.2023
  [SRO] MLP (local-env) ... MAE=0.1305  R²=0.3438
  [SRO] RandomForest ... MAE=0.1260  R²=0.3718
  [Phase] Ridge (comp) ... Acc=0.6667  F1=0.4000
  [Phase] MLP (comp) ... Acc=0.6667  F1=0.4000
  [Phase] MLP (local-env) ... Acc=1.0000  F1=1.0000
  [Phase] RandomForest ... Acc=1.0000  F1=1.0000


## 5. Initialise HEAFormer

In [9]:
from src.models.transformer import HEAEncoder, MLMHead, SROHead, PhaseHead

D = 32 if QUICK else 64

encoder    = HEAEncoder(vocab=VOCAB, feat_dim=tok.feat_dim,
                         d_model=D, n_heads=2 if QUICK else 4,
                         n_layers=2 if QUICK else 3,
                         d_ff=D*2, d_emb=D//4, seed=SEED)
mlm_head   = MLMHead(D, VOCAB)
sro_head   = SROHead(D, ds['meta']['n_sro'])
phase_head = PhaseHead(D, n_classes=3)

n_params = sum(p.size for p in encoder.all_params())
print(f"Encoder parameters : {n_params:,}")
print(f"d_model={D}  feat_dim={tok.feat_dim}  sro_dim={ds['meta']['n_sro']}")

Encoder parameters : 17,648
d_model=32  feat_dim=20  sro_dim=50


## 6. MLM Pre-training

Train the encoder to predict masked element identities.  
This teaches local chemical context without property supervision.

In [10]:
from src.training.trainer import pretrain_epoch
from src.training.losses  import Adam

N_PRE = 3 if QUICK else 6

pre_opt  = Adam(lr=2e-3, wd=0)
pre_loss = []
pre_acc  = []

print(f"MLM pre-training for {N_PRE} epochs ...")
for ep in range(N_PRE):
    r = pretrain_epoch(encoder, mlm_head, seq, tr_idx, pre_opt, rng)
    pre_loss.append(r['mlm_loss'])
    pre_acc.append(r['mlm_acc'])
    print(f"  Epoch {ep+1:2d}  loss={r['mlm_loss']:.4f}  acc={r['mlm_acc']:.4f}")

# Plot
fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 3.5))
a1.plot(range(1, N_PRE+1), pre_loss, 'b-o', ms=5, lw=2)
a1.set_xlabel('Epoch'); a1.set_ylabel('MLM Loss')
a1.set_title('MLM Pre-training Loss'); a1.grid(True, ls='--', alpha=0.4)
a2.plot(range(1, N_PRE+1), pre_acc, 'r-s', ms=5, lw=2)
a2.set_xlabel('Epoch'); a2.set_ylabel('Masked-Token Accuracy')
a2.set_title('MLM Accuracy'); a2.grid(True, ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig('figures/nb_pretrain.png', dpi=130, bbox_inches='tight')
plt.show()

MLM pre-training for 3 epochs ...
  Epoch  1  loss=1.6904  acc=0.2005
  Epoch  2  loss=1.6241  acc=0.2249
  Epoch  3  loss=1.6127  acc=0.2314


## 7. Physics-Informed SRO Fine-tuning

The SRO loss combines:
- **MSE** between predicted and true Warren-Cowley α values
- **Composition normalization** penalty: $\sum_j x_j \alpha_{ij} = 0$
- **Symmetry** penalty: $x_i \alpha_{ij} = x_j \alpha_{ji}$
- **Bounds** soft penalty: $\alpha_{ij} \in [-x_j/(1-x_j),\, 1]$

In [11]:
from src.training.trainer import finetune_epoch, evaluate

N_FT = 10 if QUICK else 25

ft_opt  = Adam(lr=1e-3 if QUICK else 5e-4, wd=1e-4)
tr_loss_hist, tr_mae_hist, va_mae_hist = [], [], []

print(f"Fine-tuning for {N_FT} epochs ...")
for ep in range(N_FT):
    r_tr = finetune_epoch(encoder, sro_head, seq, tr_idx, ft_opt, rng,
                          n_shells=2, n_sp=N_SP, batch_size=8 if QUICK else 16)
    r_va = evaluate(encoder, sro_head, None, seq, va_idx)
    tr_loss_hist.append(r_tr['loss'])
    tr_mae_hist.append(r_tr['sro_mae'])
    va_mae_hist.append(r_va['sro_mae'])
    if (ep+1) % max(1, N_FT//5) == 0 or ep == 0:
        print(f"  Ep {ep+1:3d}  loss={r_tr['loss']:.4f}  "
              f"tr_MAE={r_tr['sro_mae']:.4f}  va_MAE={r_va['sro_mae']:.4f}")

Fine-tuning for 10 epochs ...
  Ep   1  loss=3.5099  tr_MAE=0.2725  va_MAE=0.2316
  Ep   2  loss=2.2552  tr_MAE=0.2166  va_MAE=0.2178
  Ep   4  loss=1.9951  tr_MAE=0.2035  va_MAE=0.2103
  Ep   6  loss=1.9481  tr_MAE=0.2014  va_MAE=0.2074
  Ep   8  loss=1.9138  tr_MAE=0.2002  va_MAE=0.2060
  Ep  10  loss=1.8078  tr_MAE=0.1948  va_MAE=0.2007


In [12]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
eps = range(1, N_FT+1)
axes[0].plot(eps, tr_loss_hist, 'b-o', ms=3, lw=1.5, label='Train')
axes[0].set_title('Fine-tune Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(True, ls='--', alpha=0.4)

axes[1].plot(eps, tr_mae_hist, 'b-o', ms=3, lw=1.5, label='Train MAE')
axes[1].plot(eps, va_mae_hist, 'r-s', ms=3, lw=1.5, label='Val MAE')
axes[1].set_title('SRO MAE'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(True, ls='--', alpha=0.4)

plt.tight_layout()
plt.savefig('figures/nb_finetune_curves.png', dpi=130, bbox_inches='tight')
plt.show()

## 8. Phase Head Training

In [13]:
from src.training.trainer import quick_phase_train
quick_phase_train(encoder, phase_head, seq, tr_idx, lr=5e-4, epochs=4, rng=rng)
print("Phase head trained.")

Phase head trained.


## 9. Test Set Evaluation

In [14]:
from src.evaluation.metrics import compute_metrics, physics_check

# Collect test predictions
sro_preds, sro_trues = [], []
ph_preds,  ph_trues  = [], []
attn_maps            = []

for idx in te_idx:
    s  = seq[idx]
    N  = len(s['token_ids'])
    enc_out, attn = encoder.forward(s['token_ids'], s['features'])
    sro_head._N   = N
    phase_head._N = N
    sro_preds.append(sro_head.forward(enc_out))
    ph_preds.append(int(phase_head.forward(enc_out).argmax()))
    sro_trues.append(s['alpha_flat'])
    ph_trues.append(s['phase_label'])
    attn_maps.append(attn[-1])   # last-layer attention

sro_preds = np.array(sro_preds)
sro_trues = np.array(sro_trues)

metrics = compute_metrics(sro_preds, sro_trues,
                           np.array(ph_preds), np.array(ph_trues),
                           split='TEST')


[TEST] N=15
  SRO  MAE=0.2011  RMSE=0.2786  R²=0.1627  r=0.4121
  Phase Acc=0.6667  F1=0.4000


In [15]:
# SRO parity plot
fp = sro_preds.ravel(); ft = sro_trues.ravel()
lim = max(abs(ft).max(), abs(fp).max()) * 1.1

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(ft, fp, alpha=0.25, s=5, c='#3C5488', rasterized=True)
ax.plot([-lim,lim], [-lim,lim], 'r--', lw=1.5, label='Ideal')
ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim); ax.set_aspect('equal')
ax.set_xlabel(r'True $\alpha_{ij}^{(m)}$', fontsize=12)
ax.set_ylabel(r'Predicted $\alpha_{ij}^{(m)}$', fontsize=12)
ax.set_title(f"HEAFormer  MAE={metrics['sro_mae']:.4f}  R²={metrics['sro_r2']:.4f}",
             fontsize=11)
ax.legend(); ax.grid(True, ls='--', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/nb_parity.png', dpi=130, bbox_inches='tight')
plt.show()

## 10. Model Comparison (Ablation)

In [16]:
from src.evaluation.baselines import frozen_transformer_baseline

frozen_bl = frozen_transformer_baseline(encoder, seq, tr_idx, te_idx)

# Compile all SRO MAE results
all_mae = {}
for nm, m in bl['sro'].items():
    if 'mae' in m: all_mae[nm] = m['mae']
all_mae['FrozenEncoder+Ridge'] = frozen_bl.get('sro_frozen+Ridge', {}).get('mae', np.nan)
all_mae['HEAFormer (ours)']    = metrics['sro_mae']

names = sorted(all_mae, key=all_mae.get)
vals  = [all_mae[n] for n in names]

fig, ax = plt.subplots(figsize=(8, max(3, 0.45*len(names))))
colors = plt.cm.Blues(np.linspace(0.35, 0.85, len(names)))
bars = ax.barh(range(len(names)), vals, color=colors, edgecolor='k', lw=0.5)
ax.set_yticks(range(len(names)))
ax.set_yticklabels(names, fontsize=10)
ax.set_xlabel('SRO MAE (lower is better)', fontsize=11)
ax.set_title('SRO Regression — Model Comparison', fontsize=12)
for bar, v in zip(bars, vals):
    ax.text(v + max(vals)*0.01, bar.get_y()+bar.get_height()/2,
            f'{v:.4f}', va='center', fontsize=9)
ax.grid(axis='x', ls='--', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/nb_ablation_sro.png', dpi=130, bbox_inches='tight')
plt.show()

  [FrozenEncoder] extracting features ... done
  [FrozenEncoder+Ridge SRO] MAE=0.1432  R²=0.3205
  [FrozenEncoder+LR Phase] Acc=0.8667  F1=0.8295


## 11. Attention Weight Analysis

If the transformer has learned SRO-relevant patterns, unlike-element
pairs (e.g. Cr–Fe in ordering scenarios) should show elevated mutual
attention relative to like-element pairs.

In [17]:
# Pick a test sample from the 'ordering' scenario if available
ord_idx = None
for i in te_idx:
    if seq[i]['scenario_id'] > 0:   # non-random
        ord_idx = i; break
if ord_idx is None:
    ord_idx = te_idx[0]

s_ord = seq[ord_idx]
enc_out, attn = encoder.forward(s_ord['token_ids'], s_ord['features'])
A_last = attn[-1]   # (n_heads, N, N)
n_show = min(24, ds['meta']['N'])
occ    = ds['occupancy'][ord_idx]
labels = [CANTOR[int(o)] for o in occ[:n_show]]

fig, axes = plt.subplots(1, min(2, A_last.shape[0]),
                          figsize=(7*min(2, A_last.shape[0]), 6))
if A_last.shape[0] == 1:
    axes = [axes]

for h_idx, ax in enumerate(axes):
    A = A_last[h_idx, :n_show, :n_show]
    im = ax.imshow(A, cmap='Blues', vmin=0, aspect='auto')
    plt.colorbar(im, ax=ax, label='Attn weight')
    ax.set_xticks(range(n_show)); ax.set_yticks(range(n_show))
    ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=6)
    ax.set_yticklabels(labels, fontsize=6)
    for tick, lbl in zip(ax.get_xticklabels(), labels):
        tick.set_color(EL_COL.get(lbl,'k'))
    for tick, lbl in zip(ax.get_yticklabels(), labels):
        tick.set_color(EL_COL.get(lbl,'k'))
    sc_name = cfg['scenarios'][s_ord['scenario_id']]
    ax.set_title(f"Attention — last layer head {h_idx+1}\nScenario: {sc_name}",
                 fontsize=10)

plt.tight_layout()
plt.savefig('figures/nb_attention.png', dpi=130, bbox_inches='tight')
plt.show()

In [18]:
# Unlike-pair vs like-pair attention (averaged over all test samples)
A_head0 = np.stack([a[0] for a in attn_maps], axis=0)   # (Nte, N, N)

like_attn, unlike_attn = [], []
for k, idx in enumerate(te_idx):
    occ_k = ds['occupancy'][idx]
    A_k   = A_head0[k]
    for i in range(len(occ_k)):
        for j in ds['shells'][i][0]:   # 1NN only
            w = A_k[min(i, A_k.shape[0]-1), min(j, A_k.shape[1]-1)]
            if occ_k[i] == occ_k[j]:
                like_attn.append(w)
            else:
                unlike_attn.append(w)

print(f"Mean attention weight:")
print(f"  Like-pair (same element): {np.mean(like_attn):.6f}")
print(f"  Unlike-pair (diff element): {np.mean(unlike_attn):.6f}")
ratio = np.mean(unlike_attn) / (np.mean(like_attn) + 1e-10)
print(f"  Unlike/Like ratio: {ratio:.3f}  "
      f"({'unlike favoured' if ratio > 1 else 'like favoured'})")

Mean attention weight:
  Like-pair (same element): 0.009901
  Unlike-pair (diff element): 0.009101
  Unlike/Like ratio: 0.919  (like favoured)


## 12. Physics Consistency Check

In [19]:
xmean = np.ones(N_SP) / N_SP
pc = physics_check(sro_preds, xmean, 2, N_SP)

print("Warren-Cowley physics constraint violations (test set):")
print(f"  Composition normalization  sum|sum_j x_j alpha_ij|  = {pc['comp_viol']:.3e}")
print(f"  Symmetry          mean|x_i alpha_ij - x_j alpha_ji| = {pc['sym_viol']:.3e}")
print(f"  Bounds violation fraction                           = {pc['bnd_frac']:.3e}")
print("\nNote: small values (~1e-3) indicate physics constraints are approximately")
print("satisfied in the predictions due to the penalty terms in the loss.")

Warren-Cowley physics constraint violations (test set):
  Composition normalization  sum|sum_j x_j alpha_ij|  = 2.102e-02
  Symmetry          mean|x_i alpha_ij - x_j alpha_ji| = 9.877e-03
  Bounds violation fraction                           = 0.000e+00

Note: small values (~1e-3) indicate physics constraints are approximately
satisfied in the predictions due to the penalty terms in the loss.


## 13. Per-Scenario MAE Breakdown

In [20]:
sc_ids = ds['scenario_ids'][te_idx]
sc_mae = {}
for si, sc in enumerate(cfg['scenarios']):
    m_ = sc_ids == si
    if m_.sum():
        sc_mae[sc] = float(np.mean(np.abs(sro_preds[m_] - sro_trues[m_])))
        print(f"  {sc:20s}  MAE={sc_mae[sc]:.4f}  N={m_.sum()}")

fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(sc_mae))
ax.bar(x, list(sc_mae.values()),
        color=plt.cm.Set2(np.linspace(0,1,len(sc_mae))),
        edgecolor='k', lw=0.5)
ax.set_xticks(x)
ax.set_xticklabels([s.replace('_','\n') for s in sc_mae.keys()], fontsize=9)
ax.set_ylabel('SRO MAE', fontsize=11)
ax.set_title('Per-scenario SRO MAE — HEAFormer', fontsize=12)
ax.grid(axis='y', ls='--', alpha=0.4)
plt.tight_layout()
plt.savefig('figures/nb_scenario_mae.png', dpi=130, bbox_inches='tight')
plt.show()

  random                MAE=0.1382  N=6
  ordering              MAE=0.2909  N=5
  cluster               MAE=0.1831  N=4


## 14. Final Results Summary

In [21]:
print("=" * 60)
print("HEAFormer — Final Results Summary")
print("=" * 60)
print(f"\nDataset: {n} supercells | {ds['meta']['N']} atoms | "
      f"{len(cfg['scenarios'])} scenarios")
print(f"Model  : d_model={D}  n_layers={2 if QUICK else 3}  n_heads={2 if QUICK else 4}")
print(f"\nSRO Regression (test set, N={len(te_idx)}):")
print(f"  MAE    = {metrics['sro_mae']:.4f}")
print(f"  RMSE   = {metrics['sro_rmse']:.4f}")
print(f"  R²     = {metrics['sro_r2']:.4f}")
print(f"  r      = {metrics['sro_pearson']:.4f}")
print(f"\nPhase Classification:")
print(f"  Accuracy = {metrics['phase_acc']:.4f}")
print(f"  F1 macro = {metrics['phase_f1']:.4f}")
print(f"\nPhysics violations:")
print(f"  Comp normalisation: {pc['comp_viol']:.3e}")
print(f"  Symmetry:           {pc['sym_viol']:.3e}")
print(f"  Bounds fraction:    {pc['bnd_frac']:.3e}")
print("\nNote: in this small-N regime the RandomForest baseline"
      " achieves lower MAE.")
print("The transformer advantage is expected to emerge with:")
print("  (a) larger datasets (>1000 samples per scenario)")
print("  (b) more fine-tuning epochs (>50)")
print("  (c) cross-composition transfer tasks")

HEAFormer — Final Results Summary

Dataset: 90 supercells | 108 atoms | 3 scenarios
Model  : d_model=32  n_layers=2  n_heads=2

SRO Regression (test set, N=15):
  MAE    = 0.2011
  RMSE   = 0.2786
  R²     = 0.1627
  r      = 0.4121

Phase Classification:
  Accuracy = 0.6667
  F1 macro = 0.4000

Physics violations:
  Comp normalisation: 2.102e-02
  Symmetry:           9.877e-03
  Bounds fraction:    0.000e+00

Note: in this small-N regime the RandomForest baseline achieves lower MAE.
The transformer advantage is expected to emerge with:
  (a) larger datasets (>1000 samples per scenario)
  (b) more fine-tuning epochs (>50)
  (c) cross-composition transfer tasks


## 15. Save All Figures

In [ ]:
import glob
figs = sorted(glob.glob('figures/*.png'))
print(f"Figures saved ({len(figs)} total):")
for f in figs:
    sz = os.path.getsize(f) / 1024
    print(f"  {f}  ({sz:.1f} kB)")

Figures saved (23 total):
  figures\F1_architecture.png  (70.6 kB)
  figures\F2_tokenization.png  (34.5 kB)
  figures\F3_parity_HEAFormer.png  (53.9 kB)
  figures\F3_parity_TestModel.png  (53.1 kB)
  figures\F4_attn_L2_H1.png  (47.1 kB)
  figures\F5_ablation_acc_Phase.png  (36.6 kB)
  figures\F5_ablation_mae_SRO.png  (41.2 kB)
  figures\F5_ablation_r2_SRO.png  (44.8 kB)
  figures\F6_scenario_mae.png  (33.8 kB)
  figures\F7_training_curves.png  (95.7 kB)
  figures\F8_sro_matrix_sh1_cluster.png  (51.6 kB)
  figures\F8_sro_matrix_sh1_ordering.png  (60.7 kB)
  figures\F8_sro_matrix_sh1_random.png  (51.6 kB)
  figures\F8_sro_matrix_sh1_test.png  (54.1 kB)
  figures\F9_confusion.png  (43.2 kB)
  figures\nb_ablation_sro.png  (36.4 kB)
  figures\nb_attention.png  (44.2 kB)
  figures\nb_finetune_curves.png  (67.4 kB)
  figures\nb_parity.png  (59.0 kB)
  figures\nb_pretrain.png  (62.0 kB)
  figures\nb_scenario_mae.png  (27.7 kB)
  figures\nb_sro_matrices.png  (85.3 kB)
  figures\nb_tokenization.

: 